In [26]:
#Definicion del tipo de problema (variable continua) y como se representa una solucion (continua tambien)
from jmetal.core.problem import FloatProblem
from jmetal.core.solution import FloatSolution

#Utilizar un algoritmo genetico monoobjetivo
from jmetal.algorithm.singleobjective.genetic_algorithm import GeneticAlgorithm

#Operadores geneticos: crossover y mutacion
from jmetal.operator.crossover import SBXCrossover
from jmetal.operator.mutation import PolynomialMutation

#Como detener el algoritmo: iteraciones
from jmetal.util.termination_criterion import StoppingByEvaluations

#sklearn: Modelo: KNN Evaluación; validación cruzada; Dataset: Iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

#Definicion del problema: Encontrar el mejor valor de k para KNN
class KNNProblem(FloatProblem):

    def __init__(self):
        super().__init__()

        #Establecer valores minimos y maximos para el vector
        self.lower_bound = [1]
        self.upper_bound = [20]

        #Cargar en memoria el dataset
        self.data = load_iris()

    #Metodos obligatorios de jMetal
    #Definir el numero de variables: Solo se optimizará una variable: k
    def number_of_variables(self):
        return 1
        
    #Definir el numero de objetivos(en este caso es monoobjetivo): Se quiere optimizar accuracy
    def number_of_objectives(self):
        return 1
    #Definir el numero de restricciones (en este caso no hay)
    def number_of_constraints(self):
        return 0

    #Definir metodo evaluate: Es lo que el algoritmo genético intenta optimizar.
    def evaluate(self, solution: FloatSolution):

        #Convierte el valor continuo en entero válido (evita k=0)
        k = max(1, int(round(solution.variables[0])))

        #Crea el modelo KNN con ese k
        model = KNeighborsClassifier(n_neighbors=k)

        #Hace validación cruzada (3 folds): Divide datos en 3 partes; Entrena y evalúa; Promedia el resultado
        score = cross_val_score(
            model,
            self.data.data,
            self.data.target,
            cv=3
        ).mean()

        #jMetal minimiza, como se quiere maximizar, se invierte el signo.
        solution.objectives[0] = -score
        return solution

   
    #crear solucion: Genera soluciones aleatorias entre 1 y 20 (valores seteados antes)
    def create_solution(self):
        return FloatSolution(
            self.lower_bound,
            self.upper_bound,
            self.number_of_objectives()
        )

    #Nombre del problema
    def name(self):
        return "KNN: Optimizacion de Hiperparametros"


In [28]:


# ===== ejecutar =====
#Se debe respetar la secuencia definida en jMetal

#1) Crear un problema
problem = KNNProblem()

algorithm = GeneticAlgorithm(
    problem=problem,
    population_size=10,                                                     #10 soluciones por generación
    offspring_population_size=10,                                           #10 nuevas soluciones por iteración
    mutation=PolynomialMutation(probability=0.2, distribution_index=20),    #20% de probabilidad de alterar una solución (cambiar un 1 por un 2)
    crossover=SBXCrossover(probability=0.9, distribution_index=20),         #90% de probabilidad de mezclar soluciones
    termination_criterion=StoppingByEvaluations(max_evaluations=50)         #Detener después de evaluar 50 soluciones
)

#Se ejecuta el algoritmo genetico: Se realiza:
#1) Genera soluciones
#2) Evalúa (función evaluate)
#3) Selecciona mejores
#4) Cruza y muta
#5) Repite hasta terminar
algorithm.run()

#Obtiene la mejor solución encontrada
result = algorithm.result()

print("Mejor k:", int(result.variables[0]))
print("Accuracy:", -result.objectives[0])
#En las pruebas realizadas: Mejor k: 3 Accuracy: 0.9866666666666667. Este valor cambia entre ejecuciones

[2026-04-28 22:49:54,179] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-04-28 22:49:54,180] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-04-28 22:49:54,300] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-04-28 22:49:54,301] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-04-28 22:49:56,243] [jmetal.core.algorithm] [DEBUG] Finished!


Mejor k: 3
Accuracy: 0.9866666666666667
